前一步在R中
这里是共线性限制

In [1]:
import sys
sys.path.append('..')
from path_config import *

import myfunction as mf

import warnings
warnings.filterwarnings('ignore')
# list_color = ["#ee877c", "#8bd0e3", "#6abeae", "#808eaf", "#f7bba8", "#acb4cc", "#b5e0d5", "#e86462", "#a89687"]

import pandas as pd
import numpy as np
from openpyxl import Workbook
from openpyxl.styles import PatternFill
from openpyxl.utils.dataframe import dataframe_to_rows
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats


In [ ]:
# 函数准备

def forward_select_vif(df_features_raw, one_var, max_vif=10):
    """这个跟下面的区别在于这个只能指定一个初始变量"""
    selected_vars = [one_var]
    remaining_vars = [var for var in df_features_raw.columns if var != one_var]
    vif_data = pd.DataFrame(columns=['variables', 'VIF'])

    while remaining_vars:
        temp_vif_data = pd.DataFrame(columns=['variables', 'VIF'])
        for var in remaining_vars:
            temp_df = df_features_raw[selected_vars + [var]]
            temp_vif = [variance_inflation_factor(temp_df.values, i)
                        for i in range(temp_df.shape[1])]
            
            if all(vif < max_vif for vif in temp_vif[:-1]): 
                new_row = pd.DataFrame({'variables': [var], 'VIF': [temp_vif[-1]]})
                temp_vif_data = pd.concat([temp_vif_data, new_row], ignore_index=True)
        
        temp_vif_data['VIF'] = pd.to_numeric(temp_vif_data['VIF'], errors='coerce')

        temp_vif_data = temp_vif_data[temp_vif_data['VIF'] < max_vif]
        if temp_vif_data.empty:
            break
        
        next_var = temp_vif_data.loc[temp_vif_data['VIF'].idxmin(), 'variables']
        selected_vars.append(next_var)
        remaining_vars.remove(next_var)
    
    final_df = df_features_raw[selected_vars]
    vif_data['variables'] = final_df.columns
    vif_data['VIF'] = [variance_inflation_factor(final_df.values, i) 
                       for i in range(final_df.shape[1])]
    
    return vif_data


def forward_select_vif2(df_features_raw, list_var, max_vif=10):
    """通过vif_select_var调用的话用sem开头的数据即可  直接调用的话注意 list_var是初始保留的变量组
        初始变量组只有一个元素无前置条件 但是多个得先通过count_vif函数"""
    remaining_vars = [var for var in df_features_raw.columns if var not in list_var]
    print(len(remaining_vars))
    vif_data = pd.DataFrame(columns=['variables', 'VIF'])

    while remaining_vars:
        temp_vif_data = pd.DataFrame(columns=['variables', 'VIF'])
        for var in remaining_vars:
            temp_df = df_features_raw[list_var + [var]]
            temp_vif = [variance_inflation_factor(temp_df.values, i)
                        for i in range(temp_df.shape[1])]
            
            if all(vif < max_vif for vif in temp_vif[:-1]):  
                new_row = pd.DataFrame({'variables': [var], 'VIF': [temp_vif[-1]]})
                temp_vif_data = pd.concat([temp_vif_data, new_row], ignore_index=True)
        
        temp_vif_data['VIF'] = pd.to_numeric(temp_vif_data['VIF'], errors='coerce')

        temp_vif_data = temp_vif_data[temp_vif_data['VIF'] < max_vif]
        if temp_vif_data.empty:
            break
        
        next_var = temp_vif_data.loc[temp_vif_data['VIF'].idxmin(), 'variables']
        list_var.append(next_var)
        remaining_vars.remove(next_var)
        # print(remaining_vars)
    
    final_df = df_features_raw[list_var]
    vif_data['variables'] = final_df.columns
    vif_data['VIF'] = [variance_inflation_factor(final_df.values, i) 
                       for i in range(final_df.shape[1])]
    
    return vif_data


def count_vif(df_features_raw, max_vif=10):
    """迭代计算输入的dataframe VIF 每次迭代去除VIF最大的变量 直到所有变量的VIF小于10
        返回一个dataframe 里面包含两列 variables VIF"""
    vif_data = pd.DataFrame()
    vif_data["variables"] = df_features_raw.columns

    while True:
        # 计算VIF
        vif_data["VIF"] = [variance_inflation_factor(df_features_raw.values, i) 
                        for i in range(len(df_features_raw.columns))]
        
        if vif_data['VIF'].max() > max_vif:
            max_vif_feature = vif_data.loc[vif_data['VIF'].idxmax(), 'variables']
            df_features_raw = df_features_raw.drop(max_vif_feature, axis=1)
            vif_data = vif_data[vif_data['variables'] != max_vif_feature]
            print('Remove var:',max_vif_feature)
        else:
            break
    print(vif_data.shape)
    return vif_data

def print_equation(df, str_var='NO'):
    """快速生成model中潜在变量的公式 方便写代码
        这个函数直接输出文本 返回的只有okk"""
    df = df.iloc[4:]
    df = df.drop("Meaning", axis=1)
    df.set_index('vars', inplace=True)
    dic_var = {col: [] for col in df.columns}

    for index, row in df.iterrows():
        max_val = row.abs().max()
        if max_val >= 0.3:
            max_col = row.abs().idxmax()
            dic_var[max_col].append(index)

    if str_var != 'NO':
        new_dic_var = {}
        for i, (key, value) in enumerate(dic_var.items()):
            new_key = f'{str_var}_{i}'
            new_dic_var[new_key] = value
        dic_var = new_dic_var

    for key, value in dic_var.items():
        str_value = ' + '.join(value)
        print(f'{key} =~ {str_value}')

    return "okk"


def count_variance(df_sem_data):
    """计算所有变量的方差  返回的是一个df"""
    print(df_sem_data.shape)
    mean = df_sem_data.mean()
    std_dev = df_sem_data.std()
    variance = df_sem_data.var()

    # 将结果保存到一个新的DataFrame中
    df_results = pd.DataFrame({
        'var': df_sem_data.columns,
        'mean': mean.values,
        'SD': std_dev.values,
        'variance': variance.values
    })

    # 获取df_sem_data中variance列最小的值以及最大的值
    min_variance = df_results["variance"].min()
    max_variance = df_results["variance"].max()
    print(min_variance, max_variance)
    print(max_variance/min_variance)
    # 显示结果
    return df_results


def vif_select_var(df_o, path_2_preanalysis_data, path_rfecv_data, str_describe, cv_marker='' , int_max_vif=10, int_mlr=1, int_ml=10):
    """前置条件通过R语言的packfor包的sfs+mlr算法筛选变量和rfecv_imp.ipynb筛选变量"""
    df_data = df_o.copy()
    df_mlr = pd.read_csv(path_2_preanalysis_data + 'mlr_'+str_describe+'.csv')
    df_mlr = df_mlr.sort_values(by='R2', ascending=False)
    # print(df_mlr.head())

    # total_importance = df_mlr['R2'].sum()
    # 计算每行的Importance占总Importance的百分比
    # df_mlr['Relative_Imp'] = df_mlr['R2'] / total_importance * 100
    # df_rfecv = pd.read_csv(path_rfecv_data + str_describe + '_merge_rfecv'+cv_marker+'.csv')

    # min_features = df_rfecv.loc[0, 'min_features']
    # max_model = df_rfecv.loc[0, 'model']
    # max_score = round(df_rfecv.loc[0, 'mean_test_score'],3)
    # print(f'model:{max_model} min_feature:{min_features} score:{max_score}')
    # df_ml  =pd.read_csv(path_rfecv_data + str_describe + '_rfecv_features_'+max_model+'cv'+cv_marker+'.csv')
    # df_ml = df_ml[df_ml['Rank']==1]

    list_mlr_var0 = df_mlr['variables'].values
    # list_ml_var0 = df_ml['Feature'].values
    # list_re_imp0 = list(set((set(list_mlr_var0) | set(list_ml_var0))))
    list_mlr_var1 = df_mlr['variables'][df_mlr['R2']>=(int_mlr/100)].values
    # print(f'mlr lrsw:{list_mlr_var1}')
    # list_ml_var1 = df_ml['Feature'][df_ml['Importance']>=int_ml/100].values
    # print(f'ml lrsw:{list_ml_var1}')
    # list_re_imp1 = list(set((set(list_mlr_var1) | set(list_ml_var1))))
    # print(f'select var:{len(list_re_imp1)}  var:{list_re_imp1}')
    
    # list_mlr_var2 = df_mlr['variables'][df_mlr['R2']<(int_mlr/100)].values
    # list_ml_var2 = df_ml['Feature'][df_ml['Importance']<int_ml/100].values
    # list_re_imp2 = list(set((set(list_mlr_var2) | set(list_ml_var2))))
    # print(f'other var num:{len(list_re_imp2)}')
    # sw_mark = 's5'
    # df_data = pd.read_csv(path_semdata + 'sem_sw_' + sw_mark + '_au_to_'+mark_num+'_avg.csv')

    list_re_imp0 = list_mlr_var0
    list_re_imp1 = list_mlr_var1

    df_data0 = df_data[list_re_imp0]
    df_data1 = df_data[list_re_imp1]
    # df_data2 = df_data[list_re_imp2]
    
    df_vif_sbs = count_vif(df_data1,int_max_vif)
    list_start_var = df_vif_sbs['variables'].tolist()
    print(f'start var:{list_start_var}')
    df_vif_sfs = forward_select_vif2(df_data0,list_start_var,max_vif=int_max_vif)
    list_o = df_data0.columns.to_list()
    list_s = df_vif_sfs["variables"].to_list()
    list_remove  = [i for i in list_o if i not in list_s]
    print(f'SFS select var:{list_s}')
    print(f'var num:{len(list_s)}  remove num:{len(list_remove)}')
    return df_vif_sfs



In [3]:
str_describe = 'sw'
df_data = pd.read_csv(path_part0_match + 'lr_sw_linear_z.csv')
df_vif_sfs = vif_select_var(df_data, path_part2_pre, path_part3_sw, str_describe)
print(df_vif_sfs.head())
df_vif_sfs.to_csv(path_part2_pre +str_describe+'_vif_sfs.csv', index=False)

(9, 2)
start var:['po_f_carbon', 'clt', 'ts', 'WWT_CH4', 'clothing', 'solubility', 'log_D5_5', 'pr', 'log_pKa']
20
SFS select var:['po_f_carbon', 'clt', 'ts', 'WWT_CH4', 'clothing', 'solubility', 'log_D5_5', 'pr', 'log_pKa', 'sfcWind', 'distance_to_sources', 'log_Kaw', 'density', 'log_Koc', 'fluorite_consumption', 'prsn', 'population', 'paper_consumption']
var num:18  remove num:11
     variables       VIF
0  po_f_carbon  1.101196
1          clt  3.850495
2           ts  7.016954
3      WWT_CH4  6.455142
4     clothing  3.627662


In [4]:
str_describe = 'lr_sw'
df_data = pd.read_csv(path_part0_match + 'lr_sw_linear_z.csv')
df_vif_sfs = vif_select_var(df_data, path_part2_pre, path_part3_lrsw, str_describe)
print(df_vif_sfs.head())
df_vif_sfs.to_csv(path_part2_pre +str_describe+ '_vif_sfs.csv', index=False)

Remove var: log_D5_5
(10, 2)
start var:['po_chain', 'clt', 'sw_value', 'urban', 'log_Koil_w', 'log_pKa', 'log_D7_4', 'organ_liver', 'evspsbl', 'SWD_LDF_CH4']
17
SFS select var:['po_chain', 'clt', 'sw_value', 'urban', 'log_Koil_w', 'log_pKa', 'log_D7_4', 'organ_liver', 'evspsbl', 'SWD_LDF_CH4', 'solubility', 'mrro', 'psl', 'TOTALS_CO2']
var num:14  remove num:13
    variables       VIF
0    po_chain  1.293299
1         clt  2.453310
2    sw_value  2.022847
3       urban  2.668513
4  log_Koil_w  5.831220


In [5]:
str_describe = 'lr_sw'
sw_mark = 'sw'
df_vif_lr = pd.read_csv(path_part2_pre + 'lr_sw_vif_sfs.csv')
df_vif_sw = pd.read_csv(path_part2_pre + 'sw_vif_sfs.csv')

list_vars_lr = df_vif_lr['variables'].tolist()
list_vars_sw = df_vif_sw['variables'].tolist()


print('lr_all:', len(list_vars_lr))
print('sw_all:', len(list_vars_sw))
# list_vif_select = list(set(list_vars_lr + list_vars_sw))
# print(list_vif_select)
meta_data = pd.read_csv(path_file + meta_file)

list_hue_var = meta_data['var_name'][meta_data['var_type3']==0].tolist()
list_env_var = meta_data['var_name'][meta_data['var_type3']==1].tolist()
list_pfas_var = meta_data['var_name'][meta_data['var_type3']==2].tolist()
list_sp_var = meta_data['var_name'][meta_data['var_type3']==3].tolist()


list_lr_hue = list(set(list_vars_lr) & set(list_hue_var))
list_lr_env = list(set(list_vars_lr) & set(list_env_var))
list_lr_pfas = list(set(list_vars_lr) & set(list_pfas_var))
list_lr_sp = list(set(list_vars_lr) & set(list_sp_var))

list_sw_hue = list(set(list_vars_sw) & set(list_hue_var))
list_sw_env = list(set(list_vars_sw) & set(list_env_var))
list_sw_pfas = list(set(list_vars_sw) & set(list_pfas_var))
# list_sw_sp = list(set(list_vars_sw) & set(list_sp_var))

print(f'lr hue:{list_lr_hue}')
print(f'lr env:{list_lr_env}')
print(f'lr pfas:{list_lr_pfas}')
print(f'lr sp:{list_lr_sp}')

print(f'sw hue:{list_sw_hue}')
print(f'sw env:{list_sw_env}')      
print(f'sw pfas:{list_sw_pfas}')
# print(f'sw sp:{list_sw_sp}')

lr_all: 14
sw_all: 18
lr hue:['urban', 'SWD_LDF_CH4', 'TOTALS_CO2', 'sw_value']
lr env:['mrro', 'clt', 'evspsbl', 'psl']
lr pfas:['log_Koil_w', 'po_chain', 'log_D7_4', 'log_pKa', 'solubility']
lr sp:['organ_liver']
sw hue:['distance_to_sources', 'paper_consumption', 'WWT_CH4', 'fluorite_consumption', 'population', 'clothing']
sw env:['clt', 'pr', 'sfcWind', 'prsn', 'ts']
sw pfas:['po_f_carbon', 'log_Koc', 'density', 'log_D5_5', 'log_Kaw', 'log_pKa', 'solubility']


lr_all: 13
sw_all: 20
lr hue:['wrap_consumption', 'TOTALS_CO2', 'sw_value']
lr env:['clt', 'psl', 'mrro']
lr pfas:['po_chain', 'solubility', 'log_Koil_w', 'density', 'log_pKa', 'log_D7_4']
lr sp:['organ_liver']
sw hue:['fluorite_consumption', 'potential_contamination', 'TOTALS_CO2', 'GDP', 'WWT_CH4', 'urban', 'paper_consumption']
sw env:['z', 'pr', 'ts', 'clt', 'psl', 'prsn', 'mrro']
sw pfas:['po_f_carbon', 'log_Koc', 'density', 'log_pKa', 'log_D7_4', 'log_Kaw']